# Phenotype query + filtering check

Smoke test for the querying/filtering half of `02_phenotype`, run against the
real CDR before trusting the full `residualize_phenotypes.ipynb` pipeline.
Uses a short, high-confidence subset of `docs/phenotype_list.tsv` (the
confirmed anthropometric concept_ids only, no `UNCONFIRMED` lipid rows) to
check:

1. BigQuery connectivity under Workbench 2.0
2. the phenotype concept_ids resolve to the expected concepts
3. the measurement pull shape (rows, distinct persons, NA rate)
4. the round-2 keep-list intersection funnel
5. sex_at_birth / age join
6. value ranges are physiologically plausible

**Every cell below prints aggregate/summary statistics only — counts, rates,
ranges — never a person-level row.** That's deliberate, not just for output
hygiene: it keeps this notebook safe to leave un-cleared if the outputs are
ever glanced at, on top of (not instead of) the repo-wide rule to clear
outputs / run `nbstripout` before committing anything run against real AoU
data (see root `README.md`).

## Compute resource

This notebook only issues small aggregate BigQuery queries and pulls a
handful of summary numbers back locally — BigQuery does the heavy lifting
server-side. Workbench 2.0's default cloud environment app size (2 CPU / 13
GB) is enough; no need to size up for this notebook. Reach for a larger
machine only for `residualize_phenotypes.ipynb` (holds the full cohort in
memory) or the GRM stages.

In [ ]:
required_pkgs <- c("dplyr", "readr", "stringr", "glue", "bigrquery", "DBI")
missing_pkgs <- required_pkgs[!sapply(required_pkgs, requireNamespace, quietly = TRUE)]
if (length(missing_pkgs) > 0) install.packages(missing_pkgs, lib = user_lib)

library(dplyr)
library(readr)
library(stringr)
library(glue)
library(bigrquery)
library(DBI)

## Connecting to the CDR under Workbench 2.0

`residualize_phenotypes.ipynb` connects via `allofus::aou_connect()` /
`aou_sql()`, whose `{CDR}` substitution is built on the legacy Researcher
Workbench's `WORKSPACE_CDR` environment variable. Whether that still resolves
correctly on the new Verily-based Workbench 2.0 is **unconfirmed** — Verily's
own docs describe a different mechanism: workspace resources (including the
CDR BigQuery dataset) are resolved by resource ID through the `wb` CLI
(`wb resolve --id=<resource-id>`) or exposed as `$WORKBENCH_<name>`
environment variables.

This notebook connects the Workbench-2.0-documented way directly via
`bigrquery`, sidestepping the unconfirmed `allofus` dependency. If this
works and `residualize_phenotypes.ipynb`'s `allofus` calls don't, switch that
notebook's `pull_phenotype()` / `pull_covariates()` to the same connection
pattern used here.

**Two different GCP projects are involved, and they're not interchangeable.**
Each Workbench 2.0 workspace maps 1:1 to its own GCP project (used for
*billing* — this is where you have `bigquery.jobs.create`), while the CDR
dataset resolved below lives in a separate, Google-managed project you can
only read from. Passing the CDR's project as the query's billing project
fails with `Access Denied ... does not have bigquery.jobs.create permission`
in that project. `BILLING_PROJECT` (your workspace's own project) and
`BQ_PROJECT`/`BQ_DATASET_NAME` (the CDR's, read-only) are resolved
separately below and must stay separate.

The exact resource ID for the CDR dataset is workspace-specific and needs the
real workbench to pin down (`wb resource list`, or the Workspace Overview
page's Google Project panel — that panel is also where `BILLING_PROJECT`
comes from). The next cell scans `Sys.getenv()` for anything
CDR/workbench-flavored as a starting point, same approach as
`submit_pca_r1.ipynb`'s ACAF env var scan.

In [ ]:
env_vars <- Sys.getenv()
env_vars[grepl("CDR|WORKBENCH", names(env_vars), ignore.case = TRUE)]

In [ ]:
CDR_RESOURCE_ID <- "PLACEHOLDER"   # from `wb resource list` on the real workbench

BQ_DATASET <- system2("wb", c("resolve", paste0("--id=", CDR_RESOURCE_ID)), stdout = TRUE)
stopifnot(length(BQ_DATASET) == 1, nzchar(BQ_DATASET))   # `wb resolve` should return one "project.dataset" line

BQ_PROJECT <- str_split(BQ_DATASET, "\\.")[[1]][1]
BQ_DATASET_NAME <- str_split(BQ_DATASET, "\\.")[[1]][2]
CDR <- BQ_DATASET   # substituted into {CDR}.<table> below, same convention as residualize_phenotypes.ipynb's SQL

# your workspace's own GCP project (1:1 per workspace) -- NOT the CDR's project above.
# gcloud is pre-authenticated to it on a Workbench 2.0 cloud environment; this is what actually
# gets billed and what needs bigquery.jobs.create, so it must never be set to BQ_PROJECT
BILLING_PROJECT <- system2("gcloud", c("config", "get-value", "project"), stdout = TRUE)
stopifnot(length(BILLING_PROJECT) == 1, nzchar(BILLING_PROJECT), BILLING_PROJECT != BQ_PROJECT)

con <- dbConnect(bigquery(), project = BQ_PROJECT, dataset = BQ_DATASET_NAME, billing = BILLING_PROJECT)

run_query <- function(sql_template, ...) dbGetQuery(con, glue(sql_template, CDR = CDR, ...))

## Inputs

- `KEEP_LIST_PATH`: round 2's ancestry-filtering keep-list (one person_id per
  line — `round2_filter.ipynb`'s output format).
- Phenotype subset: the confirmed rows of `docs/phenotype_list.tsv`, first 3
  — enough to check the query/filter logic without pulling the full panel.

In [ ]:
KEEP_LIST_PATH <- "PLACEHOLDER"   # round 2 output, one person_id per line

keep_ids_raw <- read_lines(KEEP_LIST_PATH)
# same plink-style parsing as residualize_phenotypes.ipynb: one ID per line,
# or "FID IID" space-separated -- take the last whitespace-separated field either way
keep_ids <- str_trim(keep_ids_raw) %>% str_split(" +") %>% sapply(function(x) tail(x, 1))
length(keep_ids)

In [ ]:
pheno_list <- read_tsv("../../docs/phenotype_list.tsv", col_types = cols(.default = "c")) %>%
  filter(concept_id != "UNCONFIRMED") %>%
  slice_head(n = 3)

pheno_list

## Step 1 — concept sanity check

`{CDR}.concept` is public OMOP vocabulary metadata, not participant data —
safe to print in full. Confirms each concept_id exists, is in the expected
domain, and is a standard concept before spending a bigger query on it.

In [ ]:
concept_check <- run_query("
  SELECT concept_id, concept_name, domain_id, vocabulary_id, standard_concept
  FROM {CDR}.concept
  WHERE concept_id IN ({paste(pheno_list$concept_id, collapse = ',')})
")

concept_check

## Step 2 — measurement pull funnel

For each phenotype: total measurement rows for its concept_id, distinct
persons, and the NA rate on `value_as_number` — all aggregate counts, no
person-level output.

In [ ]:
pull_funnel_row <- function(row) {
  stats <- run_query("
    SELECT
      COUNT(*) AS n_rows,
      COUNT(DISTINCT person_id) AS n_persons,
      COUNTIF(value_as_number IS NULL) AS n_null_value
    FROM {CDR}.measurement
    WHERE measurement_concept_id = {row$concept_id}
  ", row = row)

  stats %>% mutate(phenotype_name = row$phenotype_name, .before = 1)
}

pull_funnel <- bind_rows(lapply(seq_len(nrow(pheno_list)), function(i) pull_funnel_row(pheno_list[i, ])))
pull_funnel

## Step 3 — keep-list filtering + sex/age join

Most-recent-value-per-person, joined to age and `sex_at_birth_concept_id`
(the AoU-specific column, distinct from `gender_concept_id`) — same shape
`pull_phenotype()` returns in `residualize_phenotypes.ipynb`. Reports the
funnel from raw pull down to the keep-list intersection, plus the
sex_at_birth category breakdown (aggregate counts only).

In [ ]:
REFERENCE_DATE <- "PLACEHOLDER"   # e.g. "2024-01-01"

pull_and_filter <- function(row) {
  query <- "
    WITH demographics AS (
      SELECT
        person_id,
        DATE_DIFF(DATE '{REFERENCE_DATE}', DATE(birth_datetime), YEAR) AS age,
        CASE
          WHEN sex_at_birth_concept_id = 45878463 THEN 'Female'
          WHEN sex_at_birth_concept_id = 45880669 THEN 'Male'
          ELSE 'Other'
        END AS sex_at_birth
      FROM {CDR}.person
    ),
    measurements AS (
      SELECT
        person_id,
        value_as_number AS phenotype,
        ROW_NUMBER() OVER (PARTITION BY person_id ORDER BY measurement_date DESC) AS rn
      FROM {CDR}.measurement
      WHERE measurement_concept_id = {row$concept_id}
        AND value_as_number IS NOT NULL
    )
    SELECT d.person_id, m.phenotype, d.age, d.sex_at_birth
    FROM demographics d
    INNER JOIN measurements m ON d.person_id = m.person_id
    WHERE m.rn = 1
  "
  df <- run_query(query, row = row) %>%
    mutate(person_id = as.character(person_id))

  df_filtered <- df %>% filter(person_id %in% keep_ids)

  list(name = row$phenotype_name, df = df, df_filtered = df_filtered)
}

pulls <- lapply(seq_len(nrow(pheno_list)), function(i) pull_and_filter(pheno_list[i, ]))
names(pulls) <- pheno_list$phenotype_name

filter_funnel <- bind_rows(lapply(pulls, function(p) {
  tibble(
    phenotype_name = p$name,
    n_before_keep_list = nrow(p$df),
    n_after_keep_list = nrow(p$df_filtered)
  )
}))
filter_funnel

In [ ]:
sex_breakdown <- bind_rows(lapply(pulls, function(p) {
  p$df_filtered %>% count(sex_at_birth) %>% mutate(phenotype_name = p$name, .before = 1)
}))
sex_breakdown

## Step 4 — value range sanity check

Aggregate summary stats per phenotype (min/median/mean/max/SD, NA rate) on
the keep-list-filtered values — a quick physiological-plausibility check
before trusting the pull, not a full distribution.

In [ ]:
value_summary <- bind_rows(lapply(pulls, function(p) {
  x <- p$df_filtered$phenotype
  tibble(
    phenotype_name = p$name,
    n = sum(!is.na(x)),
    min = min(x, na.rm = TRUE),
    median = median(x, na.rm = TRUE),
    mean = mean(x, na.rm = TRUE),
    max = max(x, na.rm = TRUE),
    sd = sd(x, na.rm = TRUE)
  )
}))
value_summary

## Summary

If `concept_check` matched the expected concepts, `filter_funnel` shows a
sane retention rate against the keep-list, and `value_summary`'s ranges look
physiologically plausible (e.g. height in cm, not inches or meters) — the
query and filtering logic is validated and `residualize_phenotypes.ipynb` is
safe to run on the full phenotype list.

Still needs the real workbench before that: `CDR_RESOURCE_ID`,
`KEEP_LIST_PATH`, `REFERENCE_DATE`, and confirming whether
`residualize_phenotypes.ipynb`'s `allofus`-based connection works as-is on
Workbench 2.0 or needs switching to this notebook's `bigrquery` pattern.